## Determinação das trajetórias

Agora que sabemos identificar o número das *pads* onde houve detecção, precisamos entender como fazer a correspondência entre os números das *pads* e as coordenadas espaciais, isto é, as **posições** por onde passaram os múons. Evidentemente, a posição em termos absolutos depende de adotarmos um referencial. No entanto, veremos que a posição absoluta não nos interessa; estaremos mais interessados na **posição relativa** entre as *pads* sensibilizadas das RPC de cima e de baixo para cada evento. De todo modo, isto requer um conhecimento prévio de outras informações, como a numeração e disposição das *pads* ao longo das RPCs e seus tamanhos.

As RPCs estão numeradas igualmente de acordo com a seguinte mapa 2D em forma de matriz (vista da RPC por cima):

```python 
disposicao das pads =[
    [8, 16, 24, 32, 40, 48, 56, 64], #todos os numeros desta linha sao multiplos de 8
    [7, 15, 23, 31, 39, 47, 55, 63], 
    [6, 14, 22, 30, 38, 46, 54, 62],
    [5, 13, 21, 29, 37, 45, 53, 61], 
    [4, 12, 20, 28, 36, 44, 52, 60],
    [3, 11, 19, 27, 35, 43, 51, 59],
    [2, 10, 18, 26, 34, 42, 50, 58],
    [1,  9, 17, 25, 33, 41, 49, 57]]
    numero da pad = i*8 + j
```

Por conveniência, admitiremos que as RPCs estão dispostas horizontalmente no plano xy e espaçadas verticalmente no eixo z, com eixo x indo da esquerda pra direita, eixo y de baixo pra cima, e z saindo da tela. Observe que, como as *pads* possuem tamanho fixo, a distância (em um dos eixos x ou y) entre *pads* corresponde sempre a um **múltiplo do tamanho da *pad*** nessa direção; ou seja, se as *pads* possuem 15 cm de tamanho ao longo do eixo x, a distância no eixo x entre os centros das pads 1 e 9 é de 15 cm; entre as pads 1 e 17 é 30 cm, e assim por diante. De maneira análoga, se o tamanho das *pads* no eixo y é de 19 cm, a distância no eixo y entre as *pads* 1 e 2 é de 19 cm; entre as pads 1 e 3 é de 38 cm, e assim por diante. 

Observe que a distância no eixo x depende então das **colunas** correspondentes às *pads* e a distância no eixo y das **linhas**, de acordo com a disposição 2D apresentada. Podemos então generalizar essa relação através das expressões a seguir, onde convencionamos `Lx=15cm` o tamanho das *pads* no eixo x, `Ly=19cm` o tamanho das *pads* no eixo y, e consideramos a distância entre duas *pads* denotada por $\delta X$ e $\delta Y$, nos eixos x e y respectivamente: 

$$
\delta X = (coluna_{pad_1} - coluna_{pad_2})\times L_x \\
\delta Y = (linha_{pad_1} - linha_{pad_2})\times L_y
$$

Precisamos, então, converter o número da *pad* na linha e coluna correspondente do mapa 2D de numeração. Isso pode ser feito de diferentes maneiras, observando que cada coluna abriga oito linhas. Veja, por exemplo, a função getLineAndCol() abaixo.


In [18]:
def getLineAndCol(Npad):
    line, col = Npad, 1
    while(line > 8):
        line = line - 8
    while(Npad > 8):
        Npad = Npad - 8
        col = col + 1

    return line, col   

Teste rodar a função `getLineAndCol()` com diferentes valores de Npad e compare com o que você esperava ao olhar o mapa de disposição 2D da numeração das *pads*. Os resultados fazem sentido?

In [19]:
getLineAndCol(9)

(1, 2)

De posse do número da linha e da coluna correspondentes, podemos então calcular as distâncias nos eixos x e y, `deltaX` e `deltaY`, conforme definimos anteriormente. Com os valores de `deltaX`, de `deltaY` e da distância entre as RPCs, que denotaremos por `H`, poderemos então calcular outras duas informações: os ângulos **azimutal** e **zenital**. Para isto, precisamos considerar a geometria da trajetória, conforme ilustrado na figura a seguir:

![coordenadas](coordenadas.png)

Podemos definir, portanto, as funções `getDeltaX()`, `getDeltaY()`, `getZen()` e `getAzi()` conforme feito a seguir: 

In [20]:
from math import sqrt, atan, asin, pi

Lx = 15 #cm
Ly = 19 #cm
H = 202.6 #cm

def getDeltaX(col_pad1, col_pad2):
    return (col_pad1 - col_pad2 - 1)*Lx+2*np.random.rand()*Lx

def getDeltaY(lin_pad1, lin_pad2):
    return (lin_pad1 - lin_pad2 -1)*Ly+2*np.random.rand()*Ly

def getZen(deltaX, deltaY):
    diag = sqrt(deltaX*deltaX + deltaY*deltaY)
    return (180 * atan(diag / H)/pi)

def getAzi(deltaX, deltaY):
    diag = sqrt(deltaX*deltaX + deltaY*deltaY)
    if(diag > 0):
        azi = asin(deltaY/diag)
        if (deltaX) <= 0.:
            azi = pi - azi
        if azi < 0 and (deltaX) > 0.:
            azi = 2*pi + azi
        return 180*azi/pi
    else:
        azi=np.random.rand()*360.
        return azi

Podemos então chamar estas funções em conjunto para calcular os ângulos zenital e azimutal, dando como input apenas os índices das *pads* onde houve detecção, como feito na função `calcAngles()` a seguir:

In [21]:
import pandas as pd
import numpy as np

def calcAngles(Npad1, Npad2):
    lin_pad1, col_pad1 = getLineAndCol(Npad1)
    lin_pad2, col_pad2 = getLineAndCol(Npad2)

    deltaX = getDeltaX(col_pad1, col_pad2)
    deltaY = getDeltaY(lin_pad1, lin_pad2)

    zen = getZen(deltaX, deltaY)
    azi = getAzi(deltaX, deltaY)
    
    return azi, zen

Experimente brincar com os índices das *pads* e rodar a função:

In [22]:
calcAngles(2, 14)

(261.43608876669276, 23.721765031865914)

## O output: visualizando nossos resultados

Finalmente, podemos juntar tudo que fizemos até aqui para analisar um arquivo de dados real. 
Precisamos apenas aprender a organizar nossos resultados de modo que possamos visualizá-los, analisá-los e apresentá-los a nossos pares na comunidade acadêmica. Para isso, vamos utilizar as bibliotecas `matplotlib` e `numpy`.

Observe o código abaixo:

In [23]:
from math import sqrt, atan, asin, log10, pi
from matplotlib import pyplot as plt
import numpy as np 

Lx = 15 #cm
Ly = 19 #cm
H = 202.6 #cm

def espelhar(Npad):
    if(0<Npad<65):    
        line, col = Npad, 1
        while(line > 8):
            line = line - 8
        while(Npad > 8):
            Npad = Npad - 8
            col = col + 1
        col=8-col+1
        espelhada=(col-1)*8+line    
        return espelhada 
    else:
        return Npad


def getNpads(arquivo):
    nPads_RPC1,nPads_RPC2,nPads_RPC3=[],[],[]
    binario=open(arquivo, 'rb')   #lê o arqmyfile.binuivo binário
    array = np.fromfile(binario, dtype=np.uint8) #array para o qual os bytes do arquivo serão lidos
    tamanho=len(array) #tamanho do array lido
    indice=0 #indice usado para varrer o array
    while(indice<tamanho):     # converte de uint8_t para int comum e separa as pads de cada RPC em um array diferente
        pad1=int(array[indice+1])
        pad2=int(array[indice+2])
        nPads_RPC1.append(pad1)     
        nPads_RPC2.append(pad2)
        indice+=3
    return nPads_RPC1, nPads_RPC2


def getLineAndCol(Npad):
    line, col = Npad, 1
    while(line > 8):
        line = line - 8
    while(Npad > 8):
        Npad = Npad - 8
        col = col + 1

    return line, col   

def getDeltaX(col_pad1, col_pad2):
    return (col_pad1 - col_pad2-1)*Lx+2*np.random.rand()*Lx

def getDeltaY(lin_pad1, lin_pad2):
    return (lin_pad1 - lin_pad2-1)*Ly+2*np.random.rand()*Ly

def getZen(deltaX, deltaY):
    diag = sqrt(deltaX*deltaX + deltaY*deltaY)
    return (180 * atan(diag / H)/pi)

def getAzi(deltaX, deltaY):
    diag = sqrt(deltaX*deltaX + deltaY*deltaY)
    if(diag > 0):
        azi = asin(deltaY/diag)
        if (deltaX) <= 0.:
            azi = pi - azi
        if azi < 0 and (deltaX) > 0.:
            azi = 2*pi + azi
        return 180*azi/pi
    else:
        azi=random.rand()*360
        return azi

def calcAngles(Npad1, Npad2):
    lin_pad1, col_pad1 = getLineAndCol(Npad1)
    lin_pad2, col_pad2 = getLineAndCol(Npad2)

    deltaX = getDeltaX(col_pad1, col_pad2)
    deltaY = getDeltaY(lin_pad1, lin_pad2)

    zen = getZen(deltaX, deltaY)
    azi = getAzi(deltaX, deltaY)
    
    return azi, zen



As funções criadas anteriormente podem ser usadas para analisar dados reais, que leremos do seguinte binário:

In [24]:
nPads_RPC2, nPads_RPC1 = getNpads('EAFEXP_2024_09_28.bin')

FileNotFoundError: [Errno 2] No such file or directory: 'EAFEXP_2024_09_28.bin'

In [ ]:
# from ROOT import TH2D, TCanvas

azimutal, zenital = [], []
line_RPC1, col_RPC1 = [], []
line_RPC2, col_RPC2 = [], []
validpads1,validpads2=[],[]

# h2_top = TH2D("h2_top", "", 8, 0.5, 8.5, 8, 0.5, 8.5)

for i in range(len(nPads_RPC1)):
    nPad1=nPads_RPC1[i]
    nPad2=nPads_RPC2[i]
    if(0<nPad1<65 and 0<nPad2<65):
        validpads1.append(nPad1)
        validpads2.append(nPad2)
        line1, col1 = getLineAndCol(nPad1)
        line_RPC1.append(line1)
        col_RPC1.append(col1)
        line2, col2 = getLineAndCol(nPad2)
        line_RPC2.append(line2)
        col_RPC2.append(col2)
        azi,zen=calcAngles(nPad1,nPad2)    
        azimutal.append(azi)
        zenital.append(zen)
    # h2_top.Fill(col1, line1)
    
Npads_1, Npads_2 = np.array(validpads1), np.array(validpads2)
Azimute, Zenite = np.array(azimutal), np.array(zenital)

Agora, podemos produzir *plots* das distribuições de contagens em função do número da *pad* em cada uma das RPCs de maneira bastante simples:

In [ ]:
plt.hist(Npads_1, bins=64)
plt.ylabel("Número de contagens")
plt.xlabel("Número da pad")
plt.title("RPC superior")
plt.show()

In [ ]:
plt.hist(Npads_2, bins=64)
plt.ylabel("Número de contagens")
plt.xlabel("Número da pad")
plt.title("RPC inferior")
plt.show()

Podemos ver ainda a distribuição de eventos em 2D ao longo da RPC, como quem vê a RPC de cima. Nesse caso teremos um mapa de cores, indicado no eixo z, com o número de contagens em cada pad. Para os *plots* abaixo, uma cor mais clara indica menos eventos, enquanto uma cor mais vermelha e escura indica mais eventos, de 0 a 20000. 

In [ ]:
plt.hist2d(col_RPC1, line_RPC1, bins=[8,8], cmap="YlOrRd", vmin=0)
plt.colorbar()
plt.xlabel("Coluna")
plt.ylabel("Linha")
plt.title("Mapa de pads 2D (RPC superior)")
plt.show()

In [ ]:
plt.hist2d(col_RPC2, line_RPC2, bins=[8,8], cmap="YlOrRd", vmin=0)
plt.colorbar()
plt.xlabel("Coluna")
plt.ylabel("Linha")
plt.title("Mapa de pads 2D (RPC inferior)")
plt.show()

Vamos agora ver as distribuições em função dos ângulos azimutal e zenital. O que você espera da aparência dessas distribuições?

In [ ]:
plt.hist(Azimute, bins=180, histtype='stepfilled')
plt.ylabel("Número de contagens")
plt.xlabel("Angulo azimutal")
plt.title("Distribuicao angular (azimutal)")
plt.show()

In [ ]:
plt.hist(Zenite, bins=90)
plt.ylabel("Número de contagens")
plt.xlabel("Angulo zenital")
plt.title("Distribuicao angular (zenital)")
plt.show()

In [ ]:
blacklist_RPC_1=[]
blacklist_RPC_2=[1,36,29,37,56,53]

# Hodoscópio ideal
Agora que visualizou os dados reais, que tal comparar com uma simulação de uma RPC ideal com eficiência eletrônica 100%? As funções abaixo permitem essa simulação.


In [ ]:
from math import sqrt, atan, asin, log10, pi, cos, sin, tan
from matplotlib import pyplot as plt
import numpy as np
import random

Lx = 15 #cm
Ly = 19 #cm
borda = 0.5 #cm
H = 202.6 #cm

def getLineAndCol(Npad):
    line, col = Npad, 1
    while(line > 8):
        line = line - 8
    while(Npad > 8):
        Npad = Npad - 8
        col = col + 1
    return line, col  

def getNpad(x,y,borda):
    yposinpad=y-int(y/Ly)*Ly
    xposinpad=x-int(x/Lx)*Lx
    if (borda<xposinpad<(Lx-borda) and borda<yposinpad<(Ly-borda)):
        line=int(y/Ly)+1
        col=int(x/Lx)+1
        if (line<=8 and col<=8):
            return (col-1)*8+line
        else:
            return 0
    else:
        return 0

def sortPos():
    xsort=np.random.rand()*Lx*8
    ysort=np.random.rand()*Ly*8
    return xsort,ysort


def IsInsideRPCsup(x, y):
    testpad=getNpad(x,y,borda)
    if(testpad!=0):
        if testpad in blacklist_RPC_1:
            return False
        else:
            return True 
    else:
        return False


def IsInsideRPCinf(x, y):
    testpad=getNpad(x,y,borda)
    if(testpad!=0):
        if testpad in blacklist_RPC_2:
            return False
        else:
            return True  
    else:
        return False

def IsInsideRPC(x, y):
    return ((0.< x <= 8*Lx) and (0.<= y <= 8*Ly))
    
def calcPosInf(x_sup,y_sup,zen,azi,H):
        zen_tan=tan(zen)           
        slopex=cos(azi)*zen_tan
        slopey=sin(azi)*zen_tan
        x_inf=x_sup+slopex*H
        y_inf=y_sup+slopey*H
        return x_inf,y_inf
    
def sortcos2():
    while True:
        x=np.random.rand()*pi/2. #zenite indo de 0 até 90°
        y=np.random.rand()*2./(3.*np.sqrt(3.))
        fx=sin(x)*cos(x)**2.
        if(y<fx):
            return x
        
def sortuniform():
    while True:
        x=np.random.rand()*pi/2. #zenite indo de 0 até 90°
        y=np.random.rand()
        fx=sin(x)
        if(y<fx):
            return x

def ideal(Nevts, Npad1_list, Npad2_list):
    ievt=0
    while(ievt<Nevts):
        valid=0
        while(valid==0):
            x_sup,y_sup=sortPos()
            if (IsInsideRPCsup(x_sup,y_sup)):
                #zen = sortuniform()
                zen = sortcos2()
                azi = np.random.rand()*2*pi
                x_inf, y_inf = calcPosInf(x_sup,y_sup,zen,azi,H)
                if (IsInsideRPCinf(x_inf,y_inf)):
                    Npad_sup=getNpad(x_sup,y_sup,borda)
                    Npad_inf=getNpad(x_inf,y_inf,borda)   
                    if (Npad_sup!=0 and Npad_inf !=0):
                        valid=1
                        ievt+=1     
                        azi_list.append(azi)
                        zen_list.append(zen)
                        Npad1_list.append(Npad_sup)
                        Npad2_list.append(Npad_inf)             

Execute o bloco abaixo com o número desejado de eventos, e após executado, re-execute os blocos anteriores usados para visualizar os dados, depois do bloco onde se lê "nPads_RPC1, nPads_RPC2 = getNpads('EAFEXP_2024_01_28.bin')" (sem executá-lo)

In [ ]:
Nevts = 1000000
nPads_RPC1, nPads_RPC2, zen_list, azi_list = [], [], [], []
ideal(Nevts, nPads_RPC1, nPads_RPC2)
Npads_1 = np.array(nPads_RPC1)
Npads_2 = np.array(nPads_RPC2)
Azimute, Zenite = np.array(azi_list), np.array(zen_list)